# KW_MMDS - Colab 1
## WordCount in Spark

### Setup

Let's setup Spark on your Colab environment.  Run the cell below!

In [ ]:
!pip install pyspark
!pip install -U -q PyDrive2
#the output 'xxx is not a symbolic link' will not affect your implementation or execution
#to fix 'xxx is not a symbolic link', you can comment out the lines starting from !mv xxxx
#you may need to replace xxx.11 with the correct version if other errors come up after colab update
#to get the correct version, use !ls /usr/local/lib to find out
!mv /usr/local/lib/libtbbmalloc_proxy.so.2 /usr/local/lib/libtbbmalloc_proxy.so.2.backup
!mv /usr/local/lib/libtbbmalloc.so.2 /usr/local/lib/libtbbmalloc.so.2.backup
!mv /usr/local/lib/libtbbbind_2_5.so.3 /usr/local/lib/libtbbbind_2_5.so.3.backup
!mv /usr/local/lib/libtbb.so.12 /usr/local/lib/libtbb.so.12.backup
!mv /usr/local/lib/libtbbbind_2_0.so.3 /usr/local/lib/libtbbbind_2_0.so.3.backup
!mv /usr/local/lib/libtbbbind.so.3 /usr/local/lib/libtbbbind.so.3.backup
!ln -s /usr/local/lib/libtbbmalloc_proxy.so.2.11 /usr/local/lib/libtbbmalloc_proxy.so.2
!ln -s /usr/local/lib/libtbbmalloc.so.2.11 /usr/local/lib/libtbbmalloc.so.2
!ln -s /usr/local/lib/libtbbbind_2_5.so.3.11 /usr/local/lib/libtbbbind_2_5.so.3
!ln -s /usr/local/lib/libtbb.so.12.11 /usr/local/lib/libtbb.so.12
!ln -s /usr/local/lib/libtbbbind_2_0.so.3.11 /usr/local/lib/libtbbbind_2_0.so.3
!ln -s /usr/local/lib/libtbbbind.so.3.11 /usr/local/lib/libtbbbind.so.3
# !sudo ldconfig
#If error related to the above execution occurs, you can try commenting out the above 12 lines under pip install PyDrive2 (not included)
!apt install openjdk-8-jdk-headless -qq
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
#the output 'xxx is not a symbolic link' will not affect your implementation or execution
#to fix 'xxx is not a symbolic link', you can comment out the lines starting from !mv xxxx
#you may need to replace xxx.11 with the correct version if other errors come up after colab update
#to get the correct version, use !ls /usr/local/lib to find out


The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 38 not upgraded.
Need to get 39.6 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u462-ga~us1-0ubuntu2~22.04.2_amd64.deb ...
Unpacking openjdk-8-jre-headless:amd64 (8u462-ga~us1

Now we authenticate a Google Drive client to download the file we will be processing in our Spark job.

**Make sure to follow the interactive instructions.**

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
id='1SE6k_0YukzGd5wK-E4i6mG83nydlfvSa'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('pg100.txt')

If you executed the cells above, you should be able to see the file *pg100.txt* under the "Files" tab on the left panel.

### Your task

If you run successfully the setup stage, you are ready to work on the *pg100.txt* file which contains a copy of the complete works of Shakespeare.

1. 셰익스피어 전집에서 a부터 z까지 알파벳으로 시작하는 단어의 수를 세어 RDDs1에 기록하세요. **알파벳순**으로 등장횟수를 기록하며 **대문자는 소문자로 해석합니다.** 또한 알파벳으로 시작하지 않는 단어는 무시합니다. (예: an과 apple은 a로 카운트 되고, bat와 been은 b로 카운트 됩니다.)

- 중요: RDDs1.collect()를 수행했을 때 다음과 같이 출력되어야 합니다.  
[('a', 등장횟수),  
 ('b', 등장횟수),  
  ...  
  ('z', 등장횟수)]

In [ ]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext
import pandas as pd

# create the Spark Session
spark = SparkSession.builder.getOrCreate()

# create the Spark Context
sc = spark.sparkContext

In [ ]:
# 1번 문제의 코드는 현재 셀에만 작성하세요.
lines = sc.textFile("pg100.txt")
words = lines.flatMap(lambda line: line.lower().split())
alpha_words = words.filter(lambda word: len(word) > 0 and 'a' <= word[0] <= 'z')

mapped_words = alpha_words.map(lambda word: (word[0], 1))
counts = mapped_words.reduceByKey(lambda count1, count2: count1 + count2)
RDDs1 = counts.sortByKey()
output = RDDs1.collect()
print(output)


[('a', 84836), ('b', 45455), ('c', 34567), ('d', 29713), ('e', 18697), ('f', 36814), ('g', 20782), ('h', 60563), ('i', 62167), ('j', 3339), ('k', 9418), ('l', 29569), ('m', 55676), ('n', 26759), ('o', 43494), ('p', 27759), ('q', 2377), ('r', 14265), ('s', 65705), ('t', 123602), ('u', 9170), ('v', 5728), ('w', 59597), ('x', 14), ('y', 25855), ('z', 71)]


Now, you have to work on the *2641-0.txt* file which contains 'A Room With A View' by E. M. Forster.

2. '풍경이 있는 방'에서 **소문자 c**로 시작하면서 가장 많이 등장한 단어 10개를 찾으세요. **대문자 c는 무시**하며 **많이 등장한 단어순**으로 기록해야 합니다.

- 중요: RDDs2.take(10)를 수행했을 때 다음과 같이 출력되어야 합니다.  
[('c로 시작하면서 가장 많이 등장한 단어', 등장횟수),  
 ('c로 시작하면서 두 번째로 많이 등장한 단어', 등장횟수),  
  ...  
  ('c로 시작하면서 열 번째로 많이 등장한 단어', 등장횟수)]

In [ ]:
# 풍경이 있는 방 다운로드: 2641-0.txt
room_id='1kXS_ZcsEiSdRsEwf_vIdDDJ9-RoFIcBn'
room_downloaded = drive.CreateFile({'id': room_id})
room_downloaded.GetContentFile('2641-0.txt')

In [ ]:
# 2번 문제의 코드는 현재 셀에만 작성하세요.
lines = sc.textFile('2641-0.txt')
words = lines.flatMap(lambda line: line.split())
c_words = words.filter(lambda word: word.startswith('c'))
word_pairs = c_words.map(lambda word: (word, 1))
word_counts = word_pairs.reduceByKey(lambda count1, count2: count1 + count2)
sorted_counts = word_counts.sortBy(lambda pair: pair[1], ascending=False)
top_10_list = sorted_counts.take(10)
rank_names = [
    "가장", "두 번째로", "세 번째로", "네 번째로", "다섯 번째로",
    "여섯 번째로", "일곱 번째로", "여덟 번째로", "아홉 번째로", "열 번째로"
]

formatted_list = []
for i, (word, count) in enumerate(top_10_list):
    description = f"c로 시작하면서 {rank_names[i]} 많이 등장한 단어"
    formatted_list.append((description, count))


RDDs2 = sc.parallelize(formatted_list)


print(RDDs2.collect())

[('c로 시작하면서 가장 많이 등장한 단어', 129), ('c로 시작하면서 두 번째로 많이 등장한 단어', 80), ('c로 시작하면서 세 번째로 많이 등장한 단어', 70), ('c로 시작하면서 네 번째로 많이 등장한 단어', 62), ('c로 시작하면서 다섯 번째로 많이 등장한 단어', 44), ('c로 시작하면서 여섯 번째로 많이 등장한 단어', 37), ('c로 시작하면서 일곱 번째로 많이 등장한 단어', 34), ('c로 시작하면서 여덟 번째로 많이 등장한 단어', 29), ('c로 시작하면서 아홉 번째로 많이 등장한 단어', 24), ('c로 시작하면서 열 번째로 많이 등장한 단어', 23)]


### 제출 방법 (중요)
상기 메뉴에서 파일 - 다운로드 - .py 다운로드로 다운로드한 파일을 KLAS에 제출해 주세요.